In [32]:
#Loading modules
import pandas as pd
import duckdb as db
import numpy as np
import scipy.stats as stats

In [33]:
#loading abbattoir lesions and contact farms dataframe

adf = db.query("SELECT * FROM './original_data/adf.parquet'").to_df()

adf

,vig_id,vig_date,result_date,bovine_id,farm_id,result,contact_order,last_contact,exposure_duration,herd_size
0,2622021,2021-10-15 13:16:22.0000000,2021-11-16,B1,F1,True,2,2013-03-10 00:00:00.000,575,3044.360000
1,1442022,2022-06-08 08:57:34.0000000,2022-06-23,B2,F2,False,3,2012-04-18 00:00:00.000,3529,6.350524
2,3712023,2023-07-14 12:26:07.0000000,2023-07-26,B3,F3,True,5,2022-09-02 00:00:00.000,1321,92.512491
3,3712023,2023-07-14 12:26:07.0000000,2023-07-26,B3,F4,True,4,2022-11-04 00:00:00.000,55,12.000000
4,4972024,2024-03-11 09:12:45.0000000,2024-03-20,B4,F5,True,2,2012-04-21 00:00:00.000,1273,39.860173
...,...,...,...,...,...,...,...,...,...,...
2291,7152025,2025-02-26 13:11:26.0000000,2025-03-05,B730,F554,False,2,2025-02-02 00:00:00.000,47,492.212766
2292,7122025,2025-02-24 10:22:59.0000000,2025-03-05,B436,F1845,True,3,2025-02-18 00:00:00.000,2,658.000000
2293,7122025,2025-02-24 10:22:59.0000000,2025-03-05,B436,F1846,True,6,2025-01-27 00:00:00.000,7,680.428571
2294,6912025,2025-02-11 14:12:15.0000000,2025-03-01,B732,F1847,True,3,2016-08-12 00:00:00.000,1092,168.654762


In [34]:
# Adding the route size as a variable route_size

rdf = db.query("""
            SELECT
                a.bovine_id,
                COUNT(a.farm_id) as route_size
            FROM adf a
            GROUP BY
            a.bovine_id
""").df()

adf = db.query("""
                SELECT
                    adf.*,
                    rdf.route_size

                FROM adf
                    JOIN rdf on rdf.bovine_id = adf.bovine_id
               """).df()
adf

,vig_id,vig_date,result_date,bovine_id,farm_id,result,contact_order,last_contact,exposure_duration,herd_size,route_size
0,2622021,2021-10-15 13:16:22.0000000,2021-11-16,B1,F1,True,2,2013-03-10 00:00:00.000,575,3044.360000,2
1,1442022,2022-06-08 08:57:34.0000000,2022-06-23,B2,F2,False,3,2012-04-18 00:00:00.000,3529,6.350524,4
2,3712023,2023-07-14 12:26:07.0000000,2023-07-26,B3,F3,True,5,2022-09-02 00:00:00.000,1321,92.512491,7
3,3712023,2023-07-14 12:26:07.0000000,2023-07-26,B3,F4,True,4,2022-11-04 00:00:00.000,55,12.000000,7
4,4972024,2024-03-11 09:12:45.0000000,2024-03-20,B4,F5,True,2,2012-04-21 00:00:00.000,1273,39.860173,3
...,...,...,...,...,...,...,...,...,...,...,...
2291,7152025,2025-02-26 13:11:26.0000000,2025-03-05,B730,F554,False,2,2025-02-02 00:00:00.000,47,492.212766,9
2292,7122025,2025-02-24 10:22:59.0000000,2025-03-05,B436,F1845,True,3,2025-02-18 00:00:00.000,2,658.000000,7
2293,7122025,2025-02-24 10:22:59.0000000,2025-03-05,B436,F1846,True,6,2025-01-27 00:00:00.000,7,680.428571,7
2294,6912025,2025-02-11 14:12:15.0000000,2025-03-01,B732,F1847,True,3,2016-08-12 00:00:00.000,1092,168.654762,4


In [35]:
# Filtering only the confirmed positive lesions

cdf = adf.query('result == True')

cdf

,vig_id,vig_date,result_date,bovine_id,farm_id,result,contact_order,last_contact,exposure_duration,herd_size,route_size
0,2622021,2021-10-15 13:16:22.0000000,2021-11-16,B1,F1,True,2,2013-03-10 00:00:00.000,575,3044.360000,2
2,3712023,2023-07-14 12:26:07.0000000,2023-07-26,B3,F3,True,5,2022-09-02 00:00:00.000,1321,92.512491,7
3,3712023,2023-07-14 12:26:07.0000000,2023-07-26,B3,F4,True,4,2022-11-04 00:00:00.000,55,12.000000,7
4,4972024,2024-03-11 09:12:45.0000000,2024-03-20,B4,F5,True,2,2012-04-21 00:00:00.000,1273,39.860173,3
6,2162022,2022-09-20 11:57:11.0000000,2022-09-29,B6,F7,True,1,2022-08-05 00:00:00.000,45,2073.311111,4
...,...,...,...,...,...,...,...,...,...,...,...
2285,6992025,2025-02-14 11:10:09.0000000,2025-03-01,B432,F1840,True,3,2021-01-25 00:00:00.000,795,101.213836,5
2292,7122025,2025-02-24 10:22:59.0000000,2025-03-05,B436,F1845,True,3,2025-02-18 00:00:00.000,2,658.000000,7
2293,7122025,2025-02-24 10:22:59.0000000,2025-03-05,B436,F1846,True,6,2025-01-27 00:00:00.000,7,680.428571,7
2294,6912025,2025-02-11 14:12:15.0000000,2025-03-01,B732,F1847,True,3,2016-08-12 00:00:00.000,1092,168.654762,4


In [36]:
# Creating PERT distribution (Vose, D. (2008)) with lambda values of Barlow, N. D., et al. (1997)

## Barlow, N. D., et al. (1997)
lambda_min = 1.0e-5  
lambda_mode = 2.7e-5  
lambda_max = 8.0e-5 

## Number of iterations
nit = 5000
## function to generate the distribution
def gen_pert_df(min_val, mode_val, max_val, size, seed=42):
    # Initialize the generator with a fixed seed
    rng = np.random.default_rng(seed)
    
    range_val = max_val - min_val
    if range_val == 0:
        samples = np.full(size, min_val)
    else:
        alpha = 1 + 4 * (mode_val - min_val) / range_val
        beta = 1 + 4 * (max_val - mode_val) / range_val
        
        # Use rng.beta instead of np.random.beta
        samples = min_val + rng.beta(alpha, beta, size) * range_val
    
    return pd.DataFrame({
        'iteration_id': np.arange(size),
        'lambda_val': samples
    })

## Generate our pool of nit lambdas
pert_df = gen_pert_df(lambda_min, lambda_mode, lambda_max, nit,42)



In [37]:
# Applying the Exponential Dose-Response Model using 5000 Monte Carlo Iterations and ranking

db.query("""
        SELECT 
            l.iteration_id,
            c.bovine_id,
            c.farm_id,
            c.route_size,
            CASE 
            WHEN c.herd_size <= 1  THEN 0 
            ELSE 1 - exp(-1 * l.lambda_val * c.exposure_duration * ln(c.herd_size)) 
        END AS p_ij
        FROM cdf AS c
        CROSS JOIN pert_df AS l
""").to_parquet('mc_result.parquet')

In [38]:
display(db.query("SELECT * FROM 'mc_result.parquet'").shape)
db.query("SELECT * FROM 'mc_result.parquet' TABLESAMPLE(500)").df().p_ij.describe()

(7260000, 5)

count    500.000000
mean       0.110762
std        0.144622
min        0.000000
25%        0.007376
50%        0.049355
75%        0.159251
max        0.886636
Name: p_ij, dtype: float64

In [39]:
# Calculate the mean and std dev of P_ij per path across all 5k iterations
df_cv = db.query( """
SELECT 
    bovine_id,
    farm_id,
    AVG(p_ij) as mean_p,
    STDDEV(p_ij) as std_p,
    (STDDEV(p_ij) / NULLIF(AVG(p_ij), 0)) as cv_p
FROM 'mc_result.parquet'
GROUP BY bovine_id, farm_id
"""
).df()

# Check what percentage of your population has a "tight" confidence interval
stable_count = (df_cv['cv_p'] < 0.5).mean() * 100
print(f"{stable_count:.2f}% of contact farms have a CV < 0.5")

89.60% of contact farms have a CV < 0.5


In [40]:
# Final model result incorporating deterministic and simulation results

## Barlow, N. D., et al. (1997)
lambda_min = 1.0e-5  
lambda_mode = 2.7e-5  
lambda_max = 8.0e-5 

# 2. Calculate the PERT Mean
deterministic_lambda = (lambda_min + 4 * lambda_mode + lambda_max) / 6
print(f"Calculated PERT Mean Lambda: {deterministic_lambda:.4e}")

# 3. Build the final results featuring stochastic metrics vs. deterministic P
model_result = db.query(f"""
WITH stochastic_stats AS (
    SELECT 
        m.bovine_id,
        m.farm_id,
        cdf.vig_date,
        cdf.contact_order,
        cdf.last_contact,
        cdf.exposure_duration,
        cdf.herd_size,
        cdf.route_size,
        AVG(m.p_ij) as p_mean_stochastic,
        STDDEV(m.p_ij) as p_std,
        MIN(m.p_ij) as p_min,
        MAX(m.p_ij) as p_max,
        QUANTILE_CONT(m.p_ij, 0.025) as p_ci_lower,
        QUANTILE_CONT(m.p_ij, 0.975) as p_ci_upper,
        (STDDEV(m.p_ij) / NULLIF(AVG(m.p_ij), 0)) as cv
    FROM 'mc_result.parquet' m
        JOIN cdf on cdf.bovine_id = m.bovine_id
            AND m.farm_id = cdf.farm_id
    GROUP BY 1, 2, 3,4,5,6,7,8
)
SELECT 
    s.*,
    CASE 
        WHEN c.herd_size <= 1 THEN 0
        ELSE 1 - exp(-1 * {deterministic_lambda} * c.exposure_duration * ln(c.herd_size + 1)) 
        END AS p_deterministic
FROM stochastic_stats s
JOIN cdf c ON s.bovine_id = c.bovine_id AND s.farm_id = c.farm_id
"""

).df()



Calculated PERT Mean Lambda: 3.3000e-05


In [41]:
db.query("SELECT * FROM model_result").to_parquet("model_result.parquet")
model_result

,bovine_id,farm_id,vig_date,contact_order,last_contact,exposure_duration,herd_size,route_size,p_mean_stochastic,p_std,p_min,p_max,p_ci_lower,p_ci_upper,cv,p_deterministic
0,B249,F1736,2021-07-29 11:05:54.0000000,4,2020-08-23 00:00:00.000,5,254.200000,4,0.000909,0.000339,0.000282,0.002064,0.000379,0.001632,0.373392,0.000914
1,B641,F762,2023-11-29 07:41:24.0000000,1,2021-01-20 00:00:00.000,1139,530.299809,3,0.206206,0.068186,0.070155,0.413188,0.093166,0.343895,0.330670,0.210117
2,B786,F855,2021-09-02 13:18:03.0000000,1,2020-06-21 00:00:00.000,435,200.301149,1,0.072564,0.026060,0.023195,0.158008,0.031061,0.127133,0.359131,0.073323
3,B788,F449,2024-01-16 08:47:29.0000000,1,2020-06-04 00:00:00.000,5862,190.325827,1,0.610926,0.135675,0.268898,0.899269,0.343681,0.837110,0.222081,0.638090
4,B346,F1757,2021-04-20 16:27:28.0000000,1,2021-02-08 00:00:00.000,37,2428.621622,5,0.009423,0.003503,0.002932,0.021286,0.003940,0.016867,0.371731,0.009473
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1447,B468,F1806,2023-10-19 09:31:24.0000000,1,2023-06-30 00:00:00.000,110,895.245455,5,0.024220,0.008933,0.007582,0.054250,0.010181,0.043140,0.368822,0.024376
1448,B400,F1812,2025-05-16 17:37:10.0000000,4,2023-07-04 00:00:00.000,193,1441.538860,7,0.044924,0.016384,0.014188,0.099425,0.019029,0.079459,0.364708,0.045272
1449,B722,F107,2023-08-15 16:48:30.0000000,1,2023-08-14 00:00:00.000,0,1.000000,4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000
1450,B409,F1822,2024-07-30 10:46:47.0000000,1,2023-10-27 00:00:00.000,276,7602.076087,5,0.077402,0.027721,0.024794,0.168058,0.033193,0.135380,0.358145,0.078168


In [42]:
# Verifique se existe algum resíduo decimal no 1.0
print(df[df['farm_id'] == 'F1458']['herd_size'].iloc[0] == 1.0)
print(df[df['farm_id'] == 'F1458']['herd_size'].iloc[0])

NameError: name 'df' is not defined